In [2]:
import math

# ============================================================
# CONFIGURATION SECTION - CHANGE THESE VALUES AS NEEDED
# ============================================================

# Pot dimensions
POT_RADIUS_CM = 12.0  # Radius of cylindrical pot in cm

# Tank dimensions
TANK_RADIUS_CM = 20.0   # Radius of cylindrical water tank in cm
TANK_HEIGHT_CM = 45.0   # Total height of tank in cm

# CWR prediction from your ML model
CWR_MM = 35.0  # Crop Water Requirement in millimeters

# Ultrasonic sensor reading (distance from sensor at top to water surface)
# IMPORTANT: This is the DISTANCE from top, not the water level from bottom
INITIAL_ULTRASONIC_DISTANCE_CM = 15.0  # Distance from top to water surface BEFORE opening gate

# Conversion constants
MM_TO_CM = 0.1
CM3_TO_LITERS = 0.001

# ============================================================
# CALCULATION FUNCTIONS
# ============================================================

def calculate_water_volume_needed(pot_radius_cm, cwr_mm):
    """
    Calculate the volume of water needed to satisfy CWR for the pot.
    
    Args:
        pot_radius_cm: Radius of the pot in cm
        cwr_mm: Crop Water Requirement in mm
    
    Returns:
        Tuple of (volume_cm3, volume_liters)
    """
    # Convert CWR from mm to cm
    cwr_cm = cwr_mm * MM_TO_CM
    
    # Calculate volume using cylinder formula: V = π × r² × h
    volume_cm3 = math.pi * (pot_radius_cm ** 2) * cwr_cm
    
    # Convert to liters
    volume_liters = volume_cm3 * CM3_TO_LITERS
    
    return volume_cm3, volume_liters


def calculate_water_drop_in_tank(tank_radius_cm, volume_needed_cm3):
    """
    Calculate how much the water level in the tank will drop
    when the required volume is delivered.
    
    Args:
        tank_radius_cm: Radius of the tank in cm
        volume_needed_cm3: Volume of water to be delivered in cm³
    
    Returns:
        Water level drop in cm (this is how much the water surface goes DOWN)
    """
    # Using V = π × r² × h, solve for h: h = V / (π × r²)
    tank_cross_section_area = math.pi * (tank_radius_cm ** 2)
    water_level_drop_cm = volume_needed_cm3 / tank_cross_section_area
    
    return water_level_drop_cm


def calculate_target_ultrasonic_distance(initial_distance_cm, water_drop_cm):
    """
    Calculate the target ultrasonic sensor reading after irrigation.
    
    IMPORTANT: When water leaves the tank, the water level DROPS,
    so the distance from the sensor INCREASES!
    
    Args:
        initial_distance_cm: Starting ultrasonic distance (from top to water)
        water_drop_cm: How much the water level will drop
    
    Returns:
        Target ultrasonic distance in cm (will be HIGHER than initial)
    """
    # Water drops → distance from sensor increases
    target_distance_cm = initial_distance_cm + water_drop_cm
    return target_distance_cm


def ultrasonic_distance_to_water_level(ultrasonic_distance_cm, tank_height_cm):
    """
    Convert ultrasonic sensor reading (distance from top) to actual water level from bottom.
    This is just for display/understanding purposes.
    
    Args:
        ultrasonic_distance_cm: Distance measured by ultrasonic sensor from tank top
        tank_height_cm: Total height of the tank
    
    Returns:
        Actual water level from bottom in cm
    """
    water_level_from_bottom_cm = tank_height_cm - ultrasonic_distance_cm
    return water_level_from_bottom_cm


def display_summary(cwr_mm, pot_radius_cm, volume_cm3, volume_liters, 
                   initial_distance_cm, target_distance_cm, water_drop_cm,
                   tank_height_cm):
    """
    Display a formatted summary of all calculations.
    """
    # Calculate actual water levels for display
    initial_water_level = ultrasonic_distance_to_water_level(initial_distance_cm, tank_height_cm)
    target_water_level = ultrasonic_distance_to_water_level(target_distance_cm, tank_height_cm)
    
    print("\n" + "="*70)
    print("IRRIGATION CALCULATION SUMMARY")
    print("="*70)
    
    print(f"\n📊 INPUT PARAMETERS:")
    print(f"   CWR Predicted by Model: {cwr_mm} mm")
    print(f"   Pot Radius: {pot_radius_cm} cm")
    print(f"   Tank Height: {tank_height_cm} cm")
    print(f"   Tank Radius: {TANK_RADIUS_CM} cm")
    
    print(f"\n📏 ULTRASONIC SENSOR READINGS:")
    print(f"   Initial distance from sensor to water: {initial_distance_cm} cm")
    print(f"   (This means water level from bottom: {initial_water_level} cm)")
    
    print(f"\n💧 WATER VOLUME REQUIRED:")
    print(f"   Volume needed for {cwr_mm} mm CWR: {volume_cm3:.2f} cm³")
    print(f"   Volume needed: {volume_liters:.3f} liters")
    
    print(f"\n📉 TANK WATER LEVEL CHANGES:")
    print(f"   Water level will DROP by: {water_drop_cm:.2f} cm")
    print(f"   Ultrasonic distance will INCREASE by: {water_drop_cm:.2f} cm")
    print(f"   (Because sensor is at top, more distance = less water)")
    
    print(f"\n🎯 IRRIGATION CONTROL - ULTRASONIC SENSOR READINGS:")
    print(f"   ✓ Open gate when sensor reads: {initial_distance_cm} cm")
    print(f"   ✓ Close gate when sensor reads: {target_distance_cm:.2f} cm")
    print(f"   ✓ (Distance increases from {initial_distance_cm} → {target_distance_cm:.2f} cm)")
    print(f"   ✓ This will deliver exactly {cwr_mm} mm CWR to the pot")
    
    print(f"\n📊 WATER LEVELS (from bottom of tank):")
    print(f"   Before irrigation: {initial_water_level} cm")
    print(f"   After irrigation: {target_water_level:.2f} cm")
    
    print("\n" + "="*70 + "\n")


# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    
    print("\n🌾 SMART IRRIGATION SYSTEM - CWR CALCULATION")
    print("=" * 70)
    print("Note: Ultrasonic sensor mounted at TOP of tank")
    print("      Measures DISTANCE from top to water surface")
    print("=" * 70)
    
    # Step 1: Calculate water volume needed for the pot
    volume_cm3, volume_liters = calculate_water_volume_needed(
        POT_RADIUS_CM, 
        CWR_MM
    )
    
    # Step 2: Calculate how much tank water level will drop
    water_drop_cm = calculate_water_drop_in_tank(
        TANK_RADIUS_CM, 
        volume_cm3
    )
    
    # Step 3: Calculate target ultrasonic reading
    # IMPORTANT: Distance INCREASES when water level DROPS
    target_ultrasonic_distance_cm = calculate_target_ultrasonic_distance(
        INITIAL_ULTRASONIC_DISTANCE_CM, 
        water_drop_cm
    )
    
    # Step 4: Display complete summary
    display_summary(
        CWR_MM, 
        POT_RADIUS_CM, 
        volume_cm3, 
        volume_liters,
        INITIAL_ULTRASONIC_DISTANCE_CM,
        target_ultrasonic_distance_cm,
        water_drop_cm,
        TANK_HEIGHT_CM
    )
    
    # ============================================================
    # IRRIGATION PROCESS SIMULATION
    # ============================================================
    print("🔄 IRRIGATION PROCESS SIMULATION")
    print("=" * 70)
    
    initial_water_level = ultrasonic_distance_to_water_level(
        INITIAL_ULTRASONIC_DISTANCE_CM, TANK_HEIGHT_CM
    )
    target_water_level = ultrasonic_distance_to_water_level(
        target_ultrasonic_distance_cm, TANK_HEIGHT_CM
    )
    
    print(f"\n1️⃣  Initial Check:")
    print(f"   Ultrasonic sensor reads: {INITIAL_ULTRASONIC_DISTANCE_CM} cm from top")
    print(f"   Water level in tank: {initial_water_level} cm from bottom")
    print(f"   Need to deliver: {volume_liters:.3f} liters ({water_drop_cm:.2f} cm drop)")
    
    print(f"\n2️⃣  Action: OPEN GATE ✅")
    print(f"   Water starts flowing from tank to pot...")
    
    # Simulate ultrasonic readings during irrigation
    current_distance = INITIAL_ULTRASONIC_DISTANCE_CM
    step = water_drop_cm / 5  # Divide into 5 steps for simulation
    
    for i in range(1, 6):
        current_distance += step
        current_water_level = ultrasonic_distance_to_water_level(current_distance, TANK_HEIGHT_CM)
        progress = (i / 5) * 100
        print(f"\n   [{progress:.0f}%] Sensor: {current_distance:.2f} cm | Water level: {current_water_level:.2f} cm")
    
    print(f"\n3️⃣  Target Reached!")
    print(f"   Ultrasonic sensor now reads: {target_ultrasonic_distance_cm:.2f} cm")
    print(f"   Water level now: {target_water_level:.2f} cm")
    print(f"   Action: CLOSE GATE ❌")
    
    print(f"\n✅ Irrigation Complete! {CWR_MM} mm CWR delivered successfully.")
    print(f"   Water used: {volume_liters:.3f} liters")
    print(f"   Tank level dropped by: {water_drop_cm:.2f} cm")
    print(f"   Sensor reading increased by: {water_drop_cm:.2f} cm\n")
    
    # ============================================================
    # CONTROL LOGIC SUMMARY FOR RASPBERRY PI
    # ============================================================
    print("\n" + "="*70)
    print("🤖 RASPBERRY PI CONTROL LOGIC")
    print("="*70)
    print(f"\nwhile True:")
    print(f"    current_distance = read_ultrasonic_sensor()")
    print(f"    ")
    print(f"    if current_distance >= {target_ultrasonic_distance_cm:.2f}:")
    print(f"        close_gate()")
    print(f"        print('Target reached! Gate closed.')")
    print(f"        break")
    print(f"    ")
    print(f"    # Keep gate open, continue monitoring")
    print(f"    time.sleep(0.5)  # Check every 0.5 seconds")
    print("="*70 + "\n")



🌾 SMART IRRIGATION SYSTEM - CWR CALCULATION
Note: Ultrasonic sensor mounted at TOP of tank
      Measures DISTANCE from top to water surface

IRRIGATION CALCULATION SUMMARY

📊 INPUT PARAMETERS:
   CWR Predicted by Model: 35.0 mm
   Pot Radius: 12.0 cm
   Tank Height: 45.0 cm
   Tank Radius: 20.0 cm

📏 ULTRASONIC SENSOR READINGS:
   Initial distance from sensor to water: 15.0 cm
   (This means water level from bottom: 30.0 cm)

💧 WATER VOLUME REQUIRED:
   Volume needed for 35.0 mm CWR: 1583.36 cm³
   Volume needed: 1.583 liters

📉 TANK WATER LEVEL CHANGES:
   Water level will DROP by: 1.26 cm
   Ultrasonic distance will INCREASE by: 1.26 cm
   (Because sensor is at top, more distance = less water)

🎯 IRRIGATION CONTROL - ULTRASONIC SENSOR READINGS:
   ✓ Open gate when sensor reads: 15.0 cm
   ✓ Close gate when sensor reads: 16.26 cm
   ✓ (Distance increases from 15.0 → 16.26 cm)
   ✓ This will deliver exactly 35.0 mm CWR to the pot

📊 WATER LEVELS (from bottom of tank):
   Before irrig